In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from periodic_unit_helper import *

In [ ]:
int(5 / 0.7)

In [ ]:
2 ** np.arange(5)

In [ ]:
resolution = int(5 / 0.7) * 2 + 1

In [ ]:
for factor in 2**np.arange(30) + 1:


In [ ]:
factor = 2

In [ ]:
resolution = 2 ** factor + 1

In [ ]:
x = np.linspace(0, 5, resolution)

In [ ]:
y = np.linspace(0, 5, resolution)

In [ ]:
pts = np.transpose([np.tile(x, len(y)), np.repeat(y, len(x))])


In [ ]:
edges = []
for j in range(resolution):
    for i in range(resolution-1):
        edges.append([i + resolution * j, i + 1 + resolution * j])
        
for j in range(resolution ):
    for i in range(resolution - 1):
        edges.append([j + resolution * i, j + resolution * (i + 1)])
        
for j in range(resolution-1):
    for i in range(resolution - 1):
        edges.append([j + resolution * i, j + resolution * (i + 1) + 1])

In [ ]:
triArea = 100

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(pts, edges, triArea, flags="Y")

In [ ]:
use_boundary_aligned = False

In [ ]:
box = [[1, 1], [4, 4]]
wall_box_1 = get_scaled_box(np.array(box), 1)
wall_box_2 = get_scaled_box(np.array(box) - [1, 1], 1)
wall_box_3 = get_scaled_box(np.array(box) - [1, -4], 1)
wall_box_4 = get_scaled_box(np.array(box) - [-4, 1], 1)
wall_box_5 = get_scaled_box(np.array(box) - [-4, -4], 1)

marker1 = [point_in_box(pt, wall_box_1) for pt in m.vertices()]

marker2 = [point_in_box(pt, wall_box_2) or point_in_box(pt, wall_box_3) or point_in_box(pt, wall_box_4) or point_in_box(pt, wall_box_5) for pt in m.vertices()]

marker3 = [point_in_line_segment(pt[:2], np.array([[2.5, 1], [2.5, 4]])) for pt in m.vertices()]

In [ ]:
finalMarkers = np.where(np.array(marker2 if use_boundary_aligned else marker1) == 1)[0]

In [ ]:
finalMarkers = np.where(np.array(marker3) == 1)[0]

In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=5, height=5)

In [ ]:
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

visualization.plot_2d_mesh(m, pointList=finalMarkers, width=5, height=5)

In [ ]:
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

In [ ]:
fuse_boundary = False

In [ ]:
# Fuse boundary
if fuse_boundary:
    bbox = igl.bounding_box(m.vertices())

    max_x = max(bbox[0][:, 0])
    min_x = min(bbox[0][:, 0])
    max_y = max(bbox[0][:, 1])
    min_y = min(bbox[0][:, 1])

    vxs = m.vertices()
    for i, vx in enumerate(m.vertices()):
        if np.abs(vx[0] - max_x) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[0] - min_x) < 1e-6:
            fusedVtx[i] = True    
        if np.abs(vx[1] - max_y) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[1] - min_y) < 1e-6:
            fusedVtx[i] = True    

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
viewer.setCameraParams(((-0.5299577394157474, -4.281886482597975, 2.5267752065781597),
 (0.002425713937132695, 0.5079936238005038, 0.8613574136732827),
 (0.0, 0.0, 0.0)))

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
fixedVars, hessianShift = [], 1e-6

In [ ]:
ipu.visualizationTilePower = 0

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = 3

In [ ]:
np.set_printoptions(precision=4, suppress=True)

In [ ]:
benchmark.reset()

opts.niter = 100
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
benchmark.report()

In [ ]:
ipu.sheet.energy(inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
ipu.energy()

In [ ]:
ipu.energy(inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
ipu.energy(inflation.InflatableSheet.EnergyType.Pressure)

In [ ]:
ipu.periodicVolume()

In [ ]:
# viewer.saveColorizedObj("{}.obj".format("boundary_aligned" if use_boundary_aligned else "center"))

### Vibrational Mode analysis


In [ ]:
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ModalAnalysisWrapper(ipu), mtype=compute_vibrational_modes.MassMatrixType.FULL, n=16, sigma=-1e-10, fixedVars = [])

import mode_viewer, importlib
mview = mode_viewer.ModeViewer(ipu, modes, lambdas, amplitude=10)
mview.show()